In [1]:
import numpy as np
import pandas as pd
import ollama
from datetime import datetime
from tqdm.auto import tqdm

# Load main dataset

In [2]:
# load dataframe
df_kb = pd.read_csv('../data/data-kb.csv', sep='\t', dtype=str) # they are all strings!
df_kb

,pmid,elocationid,title,journal,year,author,affiliation,abstract
0,40939192,doi: 10.1097/WNF.0000000000000653,Psilocybin Use in the Autism Spectrum Disorder...,Clinical neuropharmacology,2025,"Jaime Moreno-Chaparro, Gabriela Castañeda-Mill...","Hospital Universitario Nacional de Colombia, B...",Due to the boom in the use of certain psychede...
1,40938792,doi: 10.1080/09638237.2025.2558503,Understanding the countermovement to online pr...,"Journal of mental health (Abingdon, England)",2025,"Vaughan Bell, Rhiannon White, Lucy Foulkes","Department of Experimental Psychology, Univers...",The self-presentation of psychiatric disorders...
2,40938690,doi: 10.1097/HRP.0000000000000437,"Comorbid Autism, Anxiety, and ADHD in a Preado...",Harvard review of psychiatry,2025,"Elizabeth Steuber, Jason Fogler, Oscar Bukstei...","From Harvard Medical School (Drs. Steuber, Fog...",Autism spectrum disorder (ASD) is often comorb...
3,40938348,doi: 10.1177/2161783X251378518,MazeOut Adaptive Serious Game: Evaluation of P...,Games for health journal,2025,"Alexandre Kira, Rodrigo G Pontes, Augusto K Pe...","School of Arts, Sciences and Humanities, Unive...",NaN
4,40938167,doi: 10.1080/00207578.2024.2394076,A reading of stereotypy in autism through the ...,The International journal of psycho-analysis,2025,Leandro Jofré,"Department of Clinical Psychology, Aix Marseil...",Stereotypies currently occupy an important pla...
...,...,...,...,...,...,...,...,...
2995,40246257,doi: 10.1016/j.neuroimage.2025.121217,Interbrain synchrony attenuation during a peer...,NeuroImage,2025,"I-Chun Chen, Hao-Che Hsu, Chia-Ling Chen, Meng...",Department of Physical Medicine and Rehabilita...,Young children with autism spectrum disorder (...
2996,40245451,doi: 10.1016/j.infbeh.2025.102057,Emerging sensitivity to talking mouth in infan...,Infant behavior & development,2025,"Masahiro Hata, Mingdi Xu, Yoko Hakuno, Eriko Y...","Center for Design of Future Symbiosis, Keio Un...",The talker's mouth provides significant multim...
2997,40245419,doi: 10.1097/AOG.0000000000005896,Acetaminophen in Pregnancy and Attention-Defic...,Obstetrics and gynecology,2025,Daniel W Cramer,"Department of Obstetrics, Gynecology and Repro...",NaN
2998,40245385,pii: e63378,Evaluating a Web-Based Application to Facilita...,JMIR research protocols,2025,"Eric Meyer, Hélène Sauzéon, Isabeau Saint-Supe...","Flowers team-project, Inria Research Center of...",An individual education plan (IEP) is a key el...


# Sample PMIDs, initialize ground truth dataframe

In [3]:
# set sample size
sample_size = 100
print(sample_size)

100


In [4]:
# set number of questions per PMID
num_questions_per_pmid = 5
print(num_questions_per_pmid)

5


In [5]:
# sample PMIDs
missing_abstract = df_kb['abstract'].isna()
sampled_pmids = df_kb['pmid'][missing_abstract==False].sample(n=sample_size, \
random_state=824) # sample from those with abstract
sampled_pmids.values

array(['40526590', '40801086', '40426677', '40665512', '40491884',
       '40444763', '40681849', '40594830', '40915346', '40502168',
       '40302613', '40320052', '40530068', '40288424', '40409244',
       '40889037', '40329541', '40374608', '40372564', '40694672',
       '40420626', '40397946', '40564616', '40876096', '40534337',
       '40592404', '40483526', '40772261', '40721173', '40847203',
       '40919368', '40324921', '40452496', '40800872', '40454245',
       '40413577', '40321243', '40804712', '40712591', '40527056',
       '40419562', '40267907', '40247608', '40506196', '40562982',
       '40626515', '40350701', '40896788', '40379306', '40498657',
       '40493516', '40635406', '40723082', '40366092', '40713568',
       '40496977', '40605808', '40503466', '40488557', '40307628',
       '40685381', '40801611', '40458076', '40699321', '40343357',
       '40538922', '40317352', '40330750', '40701581', '40625432',
       '40627094', '40594285', '40797185', '40410546', '405432

In [6]:
# initialize ground truth dataframe
df_synth = pd.DataFrame({'pmid' : sorted(sampled_pmids.to_list()*num_questions_per_pmid), 
                      'ollama_seed' : [i for i in range(num_questions_per_pmid)]*sample_size})
df_synth = df_synth.merge(df_kb, on=['pmid'], how='left')[['pmid', 'ollama_seed', 'abstract']]
df_synth # has seed for reproducibility

,pmid,ollama_seed,abstract
0,40247608,0,Fragile X syndrome is the most common inherite...
1,40247608,1,Fragile X syndrome is the most common inherite...
2,40247608,2,Fragile X syndrome is the most common inherite...
3,40247608,3,Fragile X syndrome is the most common inherite...
4,40247608,4,Fragile X syndrome is the most common inherite...
...,...,...,...
495,40933686,0,"Over the past decade, universities have seen a..."
496,40933686,1,"Over the past decade, universities have seen a..."
497,40933686,2,"Over the past decade, universities have seen a..."
498,40933686,3,"Over the past decade, universities have seen a..."


# Use LLM to generate synthetic questions

In [7]:
# set LLM handle
model_handle = 'llama3.2:1b'
print(model_handle)

llama3.2:1b


In [8]:
# make prompt template for synthetic quesitons
prompt_template = """
You are a layperson who is interested in autism spectrum disorders.
Write a general question about autism that can be answered by the ABSTRACT below.
Keep the question only one or two sentences long.
Just write the question itself; do not write anything else.

PASSAGE:
{abstract}
""".strip()
print(prompt_template)

You are a layperson who is interested in autism spectrum disorders.
Write a general question about autism that can be answered by the ABSTRACT below.
Keep the question only one or two sentences long.
Just write the question itself; do not write anything else.

PASSAGE:
{abstract}


In [9]:
def generate_question(abstract, seed):
    prompt_text = prompt_template.format(abstract=abstract)
    response = ollama.chat(model=model_handle, messages=[{'role' : 'user', 'content' : prompt_text}],
                          options={'seed' : seed})
    return response['message']['content'].strip()

In [10]:
# demo question generation
demo_abstract = df_synth.iloc[0]['abstract']
print(demo_abstract)
print()
print(generate_question(demo_abstract, seed=42))

Fragile X syndrome is the most common inherited form of intellectual disability and the leading monogenetic cause of autism. Studies in mouse models of autism spectrum disorders, including the 

What are some key differences between Fragile X syndrome and autism spectrum disorder?


In [11]:
def generate_questions_list(df):
    records = df.to_dict(orient='records')
    synthetic_questions = [generate_question(record['abstract'], record['ollama_seed']) \
    for record in tqdm(records)]
    return synthetic_questions

In [12]:
# generate synthetic
print(datetime.now())
df_synth['synthetic_question'] = generate_questions_list(df_synth)
print(datetime.now())

2025-09-13 01:44:45.069110


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-13 01:59:55.881902


In [13]:
# to demonstrate reproducibility, generate again
if False: # set to True to generate again and compare
    print(datetime.now())
    demo_reproducibility = pd.Series(generate_questions_list(df_synth))
    print(datetime.now())
    print((df_synth['synthetic_question']==demo_reproducibility).value_counts()) # all true if reproducible
    print(pd.concat([df_synth['synthetic_question'], demo_reproducibility], axis=1))

In [14]:
# finalize ground truth dataframe
df_synth = df_synth[['pmid', 'ollama_seed', 'synthetic_question']]
df_synth

,pmid,ollama_seed,synthetic_question
0,40247608,0,What causes a significant portion of individua...
1,40247608,1,Is Fragile X syndrome considered a type of aut...
2,40247608,2,What causes some individuals with autism to ex...
3,40247608,3,What causes the unique communication styles an...
4,40247608,4,What are the main differences between Fragile ...
...,...,...,...
495,40933686,0,What challenges do autistic individuals face w...
496,40933686,1,Can we do away with the stigma surrounding aut...
497,40933686,2,What strategies do universities need to implem...
498,40933686,3,What challenges do autistic individuals face w...


In [15]:
# look at a few questions
print('\n'.join(df_synth['synthetic_question'].to_list()[:10]))

What causes a significant portion of individuals with autism spectrum disorders to have Fragile X syndrome as their underlying genetic condition?
Is Fragile X syndrome considered a type of autism?
What causes some individuals with autism to experience difficulties with social interaction and communication?
What causes the unique communication styles and social interactions that are characteristic of individuals with autism spectrum disorder?
What are the main differences between Fragile X syndrome and other forms of intellectual disability that may also be associated with autism?
What is the underlying mechanism by which bi-allelic UGGT1 variants lead to congenital disorders of glycosylation?
How is autism diagnosed?
What are the main differences between autism and other developmental disorders in terms of their impact on an individual's brain structure and function?
What is the primary difference between individuals with bi-allelic UGGT1 variants in CDGs compared to those without thes

# Write CSV

In [16]:
# write CSV file
df_synth.to_csv('../data/data-synth-question.csv', index=False, sep='\t')

In [17]:
print(datetime.now())

2025-09-13 01:59:55.919727
